<a href="https://colab.research.google.com/github/Leopqs/Treinamento_de_IAs/blob/main/2bimestre/trabalho/algoritnoNaiveBayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# PREVISÃO DE DIABETES COM GAUSSIAN NAIVE BAYES
# BALANCEAMENTO UTILIZANDO SMOTE + TOMEK LINKS
# DIVISÃO: 80% TREINO / 20% TESTE
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve
)

from imblearn.combine import SMOTETomek

# ==========================================
# 1. CARREGAMENTO DOS DADOS
# ==========================================

df = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv", on_bad_lines='skip')

print("==========================================")
print("INFORMAÇÕES DO DATASET")
print("==========================================")

print("\nPrimeiras linhas:")
print(df.head())

print("\nDimensões:")
print(df.shape)

print("\nDistribuição original das classes:")
print(df["Diabetes_binary"].value_counts())

# ==========================================
# 2. SEPARAÇÃO DOS ATRIBUTOS E CLASSE
# ==========================================

X = df.drop("Diabetes_binary", axis=1)
y = df["Diabetes_binary"]

# ==========================================
# 3. DIVISÃO TREINO E TESTE (80/20)
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n==========================================")
print("CONJUNTOS DE TREINO E TESTE")
print("==========================================")

print("\nTreino:", X_train.shape)
print("Teste :", X_test.shape)

print("\nDistribuição original do treino:")
print(y_train.value_counts())

# ==========================================
# 4. BALANCEAMENTO COM SMOTE + TOMEK LINKS
# ==========================================

print("\n==========================================")
print("BALANCEAMENTO DOS DADOS")
print("==========================================")

smote_tomek = SMOTETomek(
    random_state=42
)

X_train_bal, y_train_bal = smote_tomek.fit_resample(
    X_train,
    y_train
)

print("\nDistribuição após SMOTE + Tomek Links:")
print(pd.Series(y_train_bal).value_counts())

# ==========================================
# 5. TREINAMENTO DO MODELO
# ==========================================

modelo_nb = GaussianNB()

modelo_nb.fit(
    X_train_bal,
    y_train_bal
)

# ==========================================
# 6. PREVISÕES
# ==========================================

y_pred = modelo_nb.predict(X_test)

y_prob = modelo_nb.predict_proba(X_test)[:, 1]

# ==========================================
# 7. MÉTRICAS DE DESEMPENHO
# ==========================================

acuracia = accuracy_score(
    y_test,
    y_pred
)

precisao = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

auc = roc_auc_score(
    y_test,
    y_prob
)

print("\n==========================================")
print("RESULTADOS")
print("==========================================")

print(f"Acurácia : {acuracia:.4f}")
print(f"Precisão : {precisao:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"AUC ROC  : {auc:.4f}")

print("\n==========================================")
print("RELATÓRIO DE CLASSIFICAÇÃO")
print("==========================================")

print(
    classification_report(
        y_test,
        y_pred
    )
)

# ==========================================
# 8. MATRIZ DE CONFUSÃO
# ==========================================

cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Matriz de Confusão")
plt.xlabel("Classe Prevista")
plt.ylabel("Classe Real")

plt.tight_layout()
plt.show()

# ==========================================
# 9. CURVA ROC
# ==========================================

fpr, tpr, thresholds = roc_curve(
    y_test,
    y_prob
)

plt.figure(figsize=(8,6))

plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"AUC = {auc:.4f}"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)

plt.title("Curva ROC")
plt.xlabel("Taxa de Falsos Positivos")
plt.ylabel("Taxa de Verdadeiros Positivos")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ==========================================
# 10. CURVA PRECISION-RECALL
# ==========================================

precision_curve, recall_curve, _ = precision_recall_curve(
    y_test,
    y_prob
)

plt.figure(figsize=(8,6))

plt.plot(
    recall_curve,
    precision_curve,
    linewidth=2
)

plt.title("Curva Precision-Recall")
plt.xlabel("Recall")
plt.ylabel("Precisão")
plt.grid(True)

plt.tight_layout()
plt.show()

# ==========================================
# 11. GRÁFICO DAS MÉTRICAS
# ==========================================

metricas = {
    "Acurácia": acuracia,
    "Precisão": precisao,
    "Recall": recall,
    "F1-Score": f1,
    "AUC": auc
}

plt.figure(figsize=(10,6))

barras = plt.bar(
    metricas.keys(),
    metricas.values()
)

plt.ylim(0,1)

plt.title("Métricas de Desempenho")

for barra in barras:
    altura = barra.get_height()

    plt.text(
        barra.get_x() + barra.get_width()/2,
        altura + 0.01,
        f"{altura:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.show()

# ==========================================
# 12. COMPARAÇÃO DA DISTRIBUIÇÃO DAS CLASSES
# ==========================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12,5)
)

y_train.value_counts().sort_index().plot(
    kind="bar",
    ax=axes[0]
)

axes[0].set_title("Antes do Balanceamento")
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Quantidade")

pd.Series(y_train_bal).value_counts().sort_index().plot(
    kind="bar",
    ax=axes[1]
)

axes[1].set_title("Após SMOTE + Tomek Links")
axes[1].set_xlabel("Classe")
axes[1].set_ylabel("Quantidade")

plt.tight_layout()
plt.show()

# ==========================================
# 13. DISTRIBUIÇÃO DAS PROBABILIDADES
# ==========================================

plt.figure(figsize=(8,6))

plt.hist(
    y_prob,
    bins=30
)

plt.title("Distribuição das Probabilidades Previstas")
plt.xlabel("Probabilidade de Diabetes")
plt.ylabel("Quantidade")

plt.tight_layout()
plt.show()

# ==========================================
# 14. COMPARAÇÃO ENTRE CLASSES REAIS E PREVISTAS
# ==========================================

comparacao = pd.DataFrame({
    "Real": y_test,
    "Previsto": y_pred
})

plt.figure(figsize=(8,5))

comparacao["Previsto"].value_counts().sort_index().plot(
    kind="bar"
)

plt.title("Distribuição das Classes Previstas")
plt.xlabel("Classe")
plt.ylabel("Quantidade")

plt.tight_layout()
plt.show()

# ==========================================
# FIM DO PROJETO
# ==========================================
#
# MÉTRICAS GERADAS:
#
# ✔ Acurácia
# ✔ Precisão
# ✔ Recall
# ✔ F1-Score
# ✔ AUC-ROC
# ✔ Relatório de Classificação
# ✔ Matriz de Confusão
# ✔ Curva ROC
# ✔ Curva Precision-Recall
# ✔ Distribuição das Classes
# ✔ Comparação Visual das Métricas
#
# TÉCNICA DE BALANCEAMENTO:
#
# ✔ SMOTE
# ✔ Tomek Links
#
# MODELO:
#
# ✔ Gaussian Naive Bayes
#
# DIVISÃO:
#
# ✔ 80% Treino
# ✔ 20% Teste
#
# ==========================================

INFORMAÇÕES DO DATASET

Primeiras linhas:
   Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1              0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2              0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3              0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4              0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      5.0  